In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv("../data/StudentPerformanceFactors.csv")

display(df.head())

In [ ]:
print("Kích thước dữ liệu:", df.shape)

print("\nDanh sách các cột:")
print(df.columns.tolist())

print("\nThông tin dữ liệu:")
df.info()

print("\nSố lượng giá trị thiếu mỗi cột:")
print(df.isnull().sum())

print("\nSố dòng trùng lặp:", df.duplicated().sum())

print("\nThống kê dữ liệu số:")
display(df.describe())

In [ ]:
missing = df.isnull().sum()

missing_summary = pd.DataFrame({
    "So_luong_thieu": missing,
    "Ty_le_thieu_%": (missing / len(df) * 100).round(2)
})

display(missing_summary[missing_summary["So_luong_thieu"] > 0])

In [ ]:
X = df.drop(columns=["Exam_Score"])
y = df["Exam_Score"]

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Kích thước X:", X.shape)
print("Kích thước y:", y.shape)

print("\nCác cột số:")
print(numeric_features)

print("\nCác cột phân loại:")
print(categorical_features)

print("\nKiểu dữ liệu của biến mục tiêu:")
print(y.dtype)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

print("Số dòng huấn luyện:", X_train.shape[0])
print("Số dòng kiểm tra:", X_test.shape[0])
print("Đã xây dựng xong Pipeline tiền xử lý.")

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
import numpy as np
import pandas as pd

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0)
}

results = []
trained_models = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    results.append({
        "Mô hình": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2
    })

    trained_models[name] = pipeline

results_df = pd.DataFrame(results)
display(results_df.round(4))

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

ridge_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", Ridge())
])

param_grid = {
    "model__alpha": [0.01, 0.1, 1, 10, 100, 1000]
}

grid_search = GridSearchCV(
    estimator=ridge_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Alpha tốt nhất:", grid_search.best_params_["model__alpha"])
print("MAE trung bình tốt nhất trên CV:", -grid_search.best_score_)

best_ridge = grid_search.best_estimator_
y_pred_ridge = best_ridge.predict(X_test)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

print("\nKết quả trên tập kiểm tra:")
print("MAE:", round(mean_absolute_error(y_test, y_pred_ridge), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_test, y_pred_ridge)), 4))
print("R2:", round(r2_score(y_test, y_pred_ridge), 4))

In [11]:
final_results = pd.DataFrame([{
    "Mô hình": "Ridge tối ưu",
    "Alpha": grid_search.best_params_["model__alpha"],
    "MAE": mean_absolute_error(y_test, y_pred_ridge),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred_ridge)),
    "R2": r2_score(y_test, y_pred_ridge)
}])

display(final_results.round(4))

,Mô hình,Alpha,MAE,RMSE,R2
0,Ridge tối ưu,10,0.4524,1.8041,0.7697
